# GeoAI Gypsum Exploration & 3D Serpentine Traverse Pipeline

An end-to-end **Remote Sensing & GeoAI** workflow built with **Google Earth Engine (GEE)** and Python for:
1. **Multi-Sensor Fusion:** Sentinel-2 SR Harmonized (NDGI & NDVI vegetation masking) + ASTER L1T TIR (`B10, B11, B12`) + Copernicus DEM GLO-30 (2024).
2. **Geomorphological & Structural Analysis:** Topographic Position Index (TPI), multi-directional hillshade, automated lineament extraction (Canny + Probabilistic Hough Transform), 3D SVD plane fitting, and Weighted PCA outcrop strike orientation.
3. **HSE & 3D Field Logistics:** Continuous zigzag (serpentine) traverse grid perpendicular to strike, steep-slope HSE flagging (`>35°`), and 3D hiking time estimation via **Tobler's Hiking Function**.
4. **Publication-Grade Cartography:** 300 DPI multi-panel dashboard, structural rose diagram, and prioritized profile maps (with optional coordinate anonymization for public sharing).

> **Data Privacy Notice:** Raw study-area shapefiles, GEE project IDs, and exact geographic coordinates have been omitted from this notebook. Set `CONFIG['ANONYMIZE_MAPS'] = True` to render maps on a local relative metric grid.

## 1. Environment Setup & Central Configuration
Install required geospatial packages, configure pipeline parameters, and authenticate with Google Earth Engine.
**Action Required:** Replace `'YOUR_GEE_PROJECT_ID'` with your own Google Earth Engine Cloud Project ID before running.

In [ ]:
!pip install -q rasterio simplekml gpxpy openpyxl

import os, shutil, json, zipfile, glob, requests
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
from scipy.ndimage import uniform_filter
from skimage.feature import canny
from skimage.transform import probabilistic_hough_line
from skimage.measure import find_contours
from shapely.geometry import LineString
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm, ListedColormap
import simplekml
import gpxpy, gpxpy.gpx
import ee
from google.colab import files

# ==============================================================================
# تنظیمات مرکزی پایپ‌لاین (Central Configuration)
# ==============================================================================
CONFIG = {
    'PROJECT_NAME': 'Gypsum_Exploration_Site',
    'GEE_PROJECT_ID': 'YOUR_GEE_PROJECT_ID',   # شناسه پروژه GEE خود را اینجا وارد کنید
    'ANONYMIZE_MAPS': True,                    # True: مخفی کردن مختصات واقعی UTM روی نقشه‌ها
    'S2_START_DATE': '2025-11-01',
    'S2_END_DATE': '2026-11-01',
    'MAX_CLOUD_PCT': 20,
    'EXPORT_SCALE_M': 10,
    'LINE_SPACING_M': 100.0,       # فاصله پروفیل‌های پیمایش (متر)
    'STATION_INTERVAL_M': 100.0,   # فاصله ایستگاه‌ها روی هر خط (متر)
    'MIN_LINE_LENGTH_M': 80.0,     # حداقل طول معتبر خط پیمایش (متر)
    'MAX_SAFE_SLOPE_DEG': 35.0,    # آستانه ایمنی شیب HSE (درجه)
    'LOGGING_MIN_PER_ST': 10.0,    # زمان برداشت در هر ایستگاه (دقیقه)
    'FIELD_STRIKE_DEG': None,      # امتداد ساحوی (در صورت نداشتن، None بگذارید تا خودکار محاسبه شود)
    'TPI_WINDOW_PX': 9,            # پنجره 90 متری در مقیاس 10 متر
    'TPI_VALLEY_THRESH': -2.5,
    'TPI_RIDGE_THRESH': 2.5,
    'NDVI_VEG_MASK': 0.2,
    'W_SENTINEL2': 0.6,
    'W_ASTER_TIR': 0.4,
    'ANOM_PERCENTILE': 80,
    'RANDOM_SEED': 42,             # برای تکرارپذیری 100% علمی
    'OUT_DIR': '/content/GeoAI_Gypsum_Final_Outputs',
    'ZIP_PATH': '/content/Gypsum_GeoAI_Outputs.zip'
}

ee.Authenticate()
ee.Initialize(project=CONFIG['GEE_PROJECT_ID'])
print("✅ اتصال به Google Earth Engine برقرار و تنظیمات مرکزی بارگذاری شد.")

## 2. Study Area (AOI) Upload & Data Audit
Upload a zipped Shapefile (`.shp`, `.shx`, `.dbf`, `.prj`) of your Area of Interest. The script validates geometry, repairs invalid polygons, and automatically estimates the appropriate metric UTM CRS.

In [ ]:
print("📂 لطفاً فایل ZIP شپ‌فایل محدوده (AOI_Shapefile.zip) را آپلود کنید:")
uploaded = files.upload()
zip_filename = list(uploaded.keys())[0]

extract_dir = "/content/study_area"
shutil.rmtree(extract_dir, ignore_errors=True)
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_filename, 'r') as z:
    z.extractall(extract_dir)

shp_files = glob.glob(os.path.join(extract_dir, "**", "*.shp"), recursive=True)
if not shp_files:
    raise FileNotFoundError("❌ هیچ فایل .shp در آرشیو آپلودشده یافت نشد.")

gdf_raw = gpd.read_file(shp_files[0])
if gdf_raw.crs is None:
    raise ValueError("❌ فایل .prj معتبر نیست و سیستم مختصات تعریف نشده است.")

invalid_count = int((~gdf_raw.is_valid).sum())
if invalid_count > 0:
    gdf_raw['geometry'] = gdf_raw.geometry.buffer(0)

utm_crs = gdf_raw.estimate_utm_crs()
gdf_utm = gdf_raw.to_crs(utm_crs)
gdf_wgs = gdf_raw.to_crs("EPSG:4326")
poly = gdf_utm.geometry.union_all()
area_km2 = float(gdf_utm.geometry.area.sum() / 1e6)

print("\n📋 گزارش ممیزی داده (Data Audit):")
print(f"• فایل: {os.path.basename(shp_files[0])} | عوارض: {len(gdf_raw)} ({gdf_raw.geom_type.unique().tolist()})")
print(f"• سیستم مختصات ورودی: {gdf_raw.crs} -> متریک: {utm_crs.name} ({utm_crs.to_string()})")
print(f"• هندسه‌های اصلاح‌شده: {invalid_count} | مساحت دقیق: {area_km2:.4f} km²")

## 3. Multi-Sensor Satellite Data Acquisition (Google Earth Engine)
Fetches and harmonizes:
- **Copernicus DEM GLO-30 (2024_1)** (30 m native, bilinear resampled)
- **Sentinel-2 SR Harmonized** (SCL cloud/shadow masked, NDVI & NDGI computed)
- **ASTER L1T TIR** (`(B10 + B12) / (2 * B11)` gypsum thermal index, 90 m native, bilinear resampled)

In [ ]:
bounds_wgs = gdf_wgs.total_bounds
aoi_ee = ee.Geometry.Rectangle([
    bounds_wgs[0] - 0.002, bounds_wgs[1] - 0.002,
    bounds_wgs[2] + 0.002, bounds_wgs[3] + 0.002
])

# 1. Copernicus DEM 2024_1 (تفکیک‌پذیری ذاتی 30m -> بازنمونه‌برداری دوخطی پیش از موزاییک)
dem_ee = (ee.ImageCollection('COPERNICUS/DEM/GLO30_2024_1')
          .select('DEM')
          .map(lambda img: img.resample('bilinear'))
          .mosaic()
          .clip(aoi_ee))

# 2. Sentinel-2 SR Harmonized (ماسک دقیق SCL + بازنمونه‌برداری دوخطی پیش از median)
def mask_s2_scl(image):
    scl = image.select('SCL')
    mask = (scl.neq(0).And(scl.neq(1)).And(scl.neq(3))
            .And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    return (image.select(['B4', 'B8', 'B11', 'B12'])
            .multiply(0.0001)
            .updateMask(mask)
            .resample('bilinear'))

s2_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(aoi_ee)
          .filterDate(CONFIG['S2_START_DATE'], CONFIG['S2_END_DATE'])
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CONFIG['MAX_CLOUD_PCT']))
          .map(mask_s2_scl))
s2_count = int(s2_col.size().getInfo())
s2_med = s2_col.median().clip(aoi_ee)

ndvi_ee = s2_med.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndgi_ee = s2_med.normalizedDifference(['B11', 'B12']).rename('NDGI')

# 3. ASTER L1T TIR (تفکیک‌پذیری ذاتی 90m -> بازنمونه‌برداری دوخطی پیش از median)
aster_col = (ee.ImageCollection('ASTER/AST_L1T_003')
             .filterBounds(aoi_ee)
             .filter(ee.Filter.lt('CLOUDCOVER', CONFIG['MAX_CLOUD_PCT']))
             .filter(ee.Filter.listContains('system:band_names', 'B10'))
             .filter(ee.Filter.listContains('system:band_names', 'B11'))
             .filter(ee.Filter.listContains('system:band_names', 'B12')))
aster_count = int(aster_col.size().getInfo())

if aster_count > 0:
    aster_med = (aster_col.select(['B10', 'B11', 'B12'])
                 .map(lambda img: img.resample('bilinear'))
                 .median()
                 .clip(aoi_ee))
    aster_gyp_ee = aster_med.expression('(B10 + B12) / (2.0 * B11 + 0.0001)', {
        'B10': aster_med.select('B10'),
        'B11': aster_med.select('B11'),
        'B12': aster_med.select('B12')
    }).rename('ASTER_GYP')
else:
    aster_gyp_ee = ee.Image.constant(0).rename('ASTER_GYP').clip(aoi_ee)

stack = dem_ee.rename('DEM').addBands([ndvi_ee, ndgi_ee, aster_gyp_ee]).toFloat()
url = stack.getDownloadURL({
    'scale': CONFIG['EXPORT_SCALE_M'],
    'crs': utm_crs.to_string(),
    'region': aoi_ee,
    'format': 'GEO_TIFF'
})

tif_path = "/content/gee_multisensor_stack.tif"
r = requests.get(url, timeout=180)
r.raise_for_status()
with open(tif_path, 'wb') as f:
    f.write(r.content)

print(f"✅ دانلود موفق: {s2_count} تصویر Sentinel-2 + {aster_count} تصویر ASTER TIR + Copernicus DEM 2024.")

## 4. Spectral–Structural Modeling & 3D Serpentine Traverse Design
Computes terrain slope, TPI, multi-directional hillshade, weighted multi-sensor gypsum prospectivity, Canny/Hough structural lineaments, 3D SVD plane fit, Weighted PCA outcrop strike axis, and continuous serpentine survey lines with Tobler 3D hiking times.

In [ ]:
with rasterio.open(tif_path) as src:
    dem_arr       = src.read(1).astype(np.float64)
    ndvi_arr      = src.read(2).astype(np.float64)
    ndgi_arr      = src.read(3).astype(np.float64)
    aster_gyp_arr = src.read(4).astype(np.float64)
    transform     = src.transform
    height, width = dem_arr.shape

dem_arr[~np.isfinite(dem_arr)] = np.nan

extent = [transform.c, transform.c + width * transform.a,
          transform.f + height * transform.e, transform.f]
xs_g = transform.c + (np.arange(width) + 0.5) * transform.a
ys_g = transform.f + (np.arange(height) + 0.5) * transform.e
minx, miny, maxx, maxy = gdf_utm.total_bounds

cell_x, cell_y = abs(transform.a), abs(transform.e)
dy, dx = np.gradient(dem_arr, cell_y, cell_x)
slope_arr = np.degrees(np.arctan(np.hypot(dx, dy)))

outside_mask = geometry_mask([poly], out_shape=(height, width), transform=transform, invert=False)
inside_mask = ~outside_mask

# 1. TPI و سایه‌روشن چندجهته
dem_filled = np.where(np.isnan(dem_arr), np.nanmean(dem_arr), dem_arr)
tpi_arr = dem_filled - uniform_filter(dem_filled, size=CONFIG['TPI_WINDOW_PX'])

def calc_hillshade(az_deg, alt_deg=45):
    az, alt = np.deg2rad(az_deg), np.deg2rad(alt_deg)
    slp = np.arctan(np.hypot(dx, dy))
    asp = np.arctan2(-dx, dy)
    return np.clip(np.sin(alt)*np.cos(slp) + np.cos(alt)*np.sin(slp)*np.cos(az - asp), 0, 1)

multi_hillshade = sum(calc_hillshade(a) for a in (45, 135, 225, 315)) / 4.0

# 2. تلفیق طیفی–حرارتی گچ
def norm_inside(arr):
    v = arr[inside_mask & (ndvi_arr < CONFIG['NDVI_VEG_MASK']) & np.isfinite(arr)]
    p2, p98 = (np.nanpercentile(v, 2), np.nanpercentile(v, 98)) if len(v) else (0, 0)
    if p98 == p2:
        return np.zeros_like(arr)
    return np.clip((arr - p2) / (p98 - p2 + 1e-6), 0, 1)

has_aster = bool(np.nanmax(np.abs(aster_gyp_arr[inside_mask])) > 0.01)
ndgi_n  = norm_inside(ndgi_arr)
aster_n = norm_inside(aster_gyp_arr) if has_aster else ndgi_n
combined_gyp = np.where(inside_mask & (ndvi_arr < CONFIG['NDVI_VEG_MASK']),
                        CONFIG['W_SENTINEL2']*ndgi_n + CONFIG['W_ASTER_TIR']*aster_n, np.nan)
gyp_p80 = float(np.nanpercentile(combined_gyp, CONFIG['ANOM_PERCENTILE']))
anom_mask = inside_mask & np.isfinite(combined_gyp) & (combined_gyp >= gyp_p80)
gyp_binary = np.where(anom_mask, 1.0, 0.0)

# 3. استخراج تکرارپذیر خطواره‌های ساختاری + برازش سه‌بعدی SVD
hs_inside = np.where(inside_mask, multi_hillshade, np.nanmean(multi_hillshade[inside_mask]))
edges = canny(hs_inside, sigma=1.5, low_threshold=0.05, high_threshold=0.15) & inside_mask
lines_hough = probabilistic_hough_line(edges, threshold=8, line_length=12, line_gap=3, rng=CONFIG['RANDOM_SEED'])

lineament_azimuths, lineament_lengths, lineament_geoms = [], [], []
for (c0, r0), (c1, r1) in lines_hough:
    x0, y0 = rasterio.transform.xy(transform, r0, c0)
    x1, y1 = rasterio.transform.xy(transform, r1, c1)
    lineament_azimuths.append(np.degrees(np.arctan2(x1 - x0, y1 - y0)) % 180.0)
    lineament_lengths.append(np.hypot(x1 - x0, y1 - y0))
    lineament_geoms.append(LineString([(x0, y0), (x1, y1)]))

if lineament_azimuths:
    rad2 = np.deg2rad(2.0 * np.array(lineament_azimuths))
    w = np.array(lineament_lengths)
    dominant_lin_strike = float((0.5 * np.degrees(np.arctan2(np.sum(w*np.sin(rad2)), np.sum(w*np.cos(rad2))))) % 180.0)
else:
    dominant_lin_strike = 90.0

contours_gyp = find_contours(gyp_binary, 0.5)
rmse_plane = np.nan
if contours_gyp:
    pts_3d = []
    for r_f, c_f in max(contours_gyp, key=len):
        r_i, c_i = int(round(r_f)), int(round(c_f))
        if 0 <= r_i < height and 0 <= c_i < width and np.isfinite(dem_arr[r_i, c_i]):
            x_c, y_c = rasterio.transform.xy(transform, r_i, c_i)
            pts_3d.append([x_c, y_c, dem_arr[r_i, c_i]])
    if len(pts_3d) >= 10:
        pts_3d = np.array(pts_3d)
        centered = pts_3d - pts_3d.mean(axis=0)
        _, _, vh = np.linalg.svd(centered)
        normal = vh[-1, :] if vh[-1, 2] >= 0 else -vh[-1, :]
        rmse_plane = float(np.sqrt(np.mean((centered @ normal)**2)))

# 4. محور امتداد رخنمون گچ (Weighted PCA)
rows_a, cols_a = np.where(anom_mask)
xs_a, ys_a = rasterio.transform.xy(transform, rows_a, cols_a)
pts_xy = np.column_stack([np.array(xs_a), np.array(ys_a)])
w_a = combined_gyp[rows_a, cols_a]
if len(w_a) < 10 or np.nansum(w_a) <= 0:
    raise ValueError("❌ داده‌های طیفی گچ معتبر نیستند (مجموع وزن‌ها صفر است). خروجی سلول ۳ را بررسی کنید.")

centroid_xy = np.average(pts_xy, axis=0, weights=w_a)
eigvals, eigvecs = np.linalg.eigh(np.cov((pts_xy - centroid_xy).T, aweights=w_a))
order = np.argsort(eigvals)[::-1]
v_strike_pca = eigvecs[:, order[0]]
if v_strike_pca[0] < 0: v_strike_pca = -v_strike_pca
outcrop_strike_az = float(np.degrees(np.arctan2(v_strike_pca[0], v_strike_pca[1])) % 180.0)

diag = np.hypot(maxx - minx, maxy - miny)
strike_raw = LineString([centroid_xy - 0.6*diag*v_strike_pca, centroid_xy + 0.6*diag*v_strike_pca]).intersection(poly)
outcrop_strike_line = max(strike_raw.geoms, key=lambda g: g.length) if strike_raw.geom_type == 'MultiLineString' else strike_raw

# 5. طراحی شبکه پیمایش پیوسته زیگزاگ (Serpentine) + لجستیک سه‌بعدی توبلر
inferred_strike_deg = (CONFIG['FIELD_STRIKE_DEG'] % 180.0) if CONFIG['FIELD_STRIKE_DEG'] is not None else dominant_lin_strike
traverse_az_deg = (inferred_strike_deg + 90.0) % 180.0
u_trav   = np.array([np.sin(np.deg2rad(traverse_az_deg)), np.cos(np.deg2rad(traverse_az_deg))])
u_strike = np.array([np.sin(np.deg2rad(inferred_strike_deg)), np.cos(np.deg2rad(inferred_strike_deg))])

def sample_raster(arr, pt):
    r, c = rasterio.transform.rowcol(transform, pt.x, pt.y)
    return float(arr[r, c]) if (0 <= r < height and 0 <= c < width and np.isfinite(arr[r, c])) else np.nan

def tobler_time_min(dd_arr, dz_arr):
    speed_kmh = np.clip(6.0 * np.exp(-3.5 * np.abs(dz_arr / (dd_arr + 1e-6) + 0.05)), 0.4, 6.0)
    return float(np.sum((dd_arr / 1000.0) / speed_kmh) * 60.0)

cx, cy = poly.centroid.x, poly.centroid.y
lines_records, stations_records, connector_geoms = [], [], []
all_route_coords = []
prev_end_pt = None
line_idx = 1

for off in np.arange(-diag/2, diag/2 + CONFIG['LINE_SPACING_M'], CONFIG['LINE_SPACING_M']):
    c_pt = np.array([cx, cy]) + off * u_strike
    clipped = LineString([c_pt - (diag/2)*u_trav, c_pt + (diag/2)*u_trav]).intersection(poly)
    if clipped.is_empty: continue
    segs = list(clipped.geoms) if clipped.geom_type == 'MultiLineString' else [clipped]
    for line in segs:
        if line.length < CONFIG['MIN_LINE_LENGTH_M']: continue

        # معکوس کردن جهت خطوط زوج برای ایجاد الگوی زیگزاگ پیوسته (Serpentine)
        if line_idx % 2 == 0:
            line = LineString(list(line.coords)[::-1])
            line_az = (traverse_az_deg + 180.0) % 360.0
            dir_label = 'N->S'
        else:
            line_az = traverse_az_deg
            dir_label = 'S->N'

        # اتصال انتهای پروفیل قبلی به ابتدای پروفیل جدید
        if prev_end_pt is not None:
            connector_geoms.append(LineString([prev_end_pt, line.coords[0]]))
        prev_end_pt = line.coords[-1]
        all_route_coords.extend(list(line.coords))

        line_id = f"L-{line_idx:02d}"
        line_idx += 1

        dense_d = np.arange(0, line.length, 10.0)
        if dense_d[-1] < line.length: dense_d = np.append(dense_d, line.length)
        elevs  = pd.Series([sample_raster(dem_arr, line.interpolate(d)) for d in dense_d]).interpolate().bfill().ffill()
        slopes = pd.Series([sample_raster(slope_arr, line.interpolate(d)) for d in dense_d]).interpolate().bfill().ffill()
        gyps   = [sample_raster(combined_gyp, line.interpolate(d)) for d in dense_d]
        tpis   = [sample_raster(tpi_arr, line.interpolate(d)) for d in dense_d]

        dd, dz = np.diff(dense_d), np.diff(elevs)
        len_3d = float(np.sum(np.hypot(dd, dz)))
        walk_min = tobler_time_min(dd, dz)

        st_dists = np.arange(0, line.length, CONFIG['STATION_INTERVAL_M'])
        if line.length - st_dists[-1] > 25.0: st_dists = np.append(st_dists, line.length)
        total_hr = (walk_min + len(st_dists) * CONFIG['LOGGING_MIN_PER_ST']) / 60.0
        max_slp  = float(np.nanmax(slopes))
        anom_pct = float(np.nanmean([1.0 if (np.isfinite(v) and v >= gyp_p80) else 0.0 for v in gyps]) * 100)
        val_pct  = float(np.nanmean([1.0 if (np.isfinite(v) and v < CONFIG['TPI_VALLEY_THRESH']) else 0.0 for v in tpis]) * 100)
        priority = 'High' if anom_pct >= 20 else ('Medium' if anom_pct >= 10 else 'Low')

        lines_records.append({
            'Line_ID': line_id, 'Direction': dir_label, 'Azimuth': round(line_az, 1),
            'Len_2D_m': round(line.length, 1), 'Len_3D_m': round(len_3d, 1),
            'Max_Slope': round(max_slp, 1),
            'HSE_Flag': f"Caution (>{int(CONFIG['MAX_SAFE_SLOPE_DEG'])}°)" if max_slp > CONFIG['MAX_SAFE_SLOPE_DEG'] else 'Safe',
            'Walk_min': round(walk_min, 1), 'Total_hr': round(total_hr, 2),
            'Valley_%': round(val_pct, 1), 'Gyp_Score': round(float(np.nanmean(gyps)), 3),
            'Anom_%': round(anom_pct, 1), 'Priority': priority, 'geometry': line
        })

        for j, d in enumerate(st_dists, start=1):
            pt_utm = line.interpolate(d)
            pt_wgs = gpd.GeoSeries([pt_utm], crs=utm_crs).to_crs("EPSG:4326").iloc[0]
            g_val, t_val = sample_raster(combined_gyp, pt_utm), sample_raster(tpi_arr, pt_utm)
            tpi_cls = 'Valley/Outcrop' if (np.isfinite(t_val) and t_val < CONFIG['TPI_VALLEY_THRESH']) else (
                      'Ridge' if (np.isfinite(t_val) and t_val > CONFIG['TPI_RIDGE_THRESH']) else 'Slope')
            stations_records.append({
                'Line_ID': line_id, 'Station_ID': f"{line_id}-ST{j:02d}", 'Chainage_m': round(d, 1),
                'UTM_E': round(pt_utm.x, 2), 'UTM_N': round(pt_utm.y, 2),
                'Latitude': round(pt_wgs.y, 6), 'Longitude': round(pt_wgs.x, 6),
                'Elev_m': round(sample_raster(dem_arr, pt_utm), 1),
                'Slope_deg': round(sample_raster(slope_arr, pt_utm), 1),
                'TPI_Class': tpi_cls, 'Gyp_Score': round(g_val, 3) if np.isfinite(g_val) else np.nan,
                'Target': 'Gypsum Anomaly' if (np.isfinite(g_val) and g_val >= gyp_p80) else 'Normal',
                'Lithology': '', 'Strike_Dip': '', 'Sample_ID': '', 'Notes': '', 'geometry': pt_utm
            })

gdf_lines = gpd.GeoDataFrame(lines_records, crs=utm_crs)
gdf_stations = gpd.GeoDataFrame(stations_records, crs=utm_crs)
gdf_connectors = gpd.GeoDataFrame({'Type': ['Profile_Connector'] * len(connector_geoms), 'geometry': connector_geoms}, crs=utm_crs)
gdf_continuous_route = gpd.GeoDataFrame([{'Route': 'Continuous_Serpentine_Traverse', 'geometry': LineString(all_route_coords)}], crs=utm_crs)
gdf_strike = gpd.GeoDataFrame([{'Feature': 'Gypsum_Outcrop_Strike_Axis',
                                'Strike_Az_deg': round(outcrop_strike_az, 1),
                                'Length_m': round(outcrop_strike_line.length, 1),
                                'geometry': outcrop_strike_line}], crs=utm_crs)

conn_len_km = float(gdf_connectors.geometry.length.sum() / 1000.0) if len(gdf_connectors) else 0.0
print(f"✅ مسیر پیوسته زیگزاگ کامل شد: {len(gdf_lines)} خط ({CONFIG['LINE_SPACING_M']}m) + {len(gdf_connectors)} خط اتصال ({conn_len_km:.2f} km) | {len(gdf_stations)} ایستگاه")
print(f"📏 طول 3D پروفیل‌ها: {gdf_lines['Len_3D_m'].sum()/1000:.2f} km | زمان خالص پروفیل‌ها: {gdf_lines['Total_hr'].sum():.1f} ساعت")
print(f"🧭 امتداد خطواره‌ها: N{dominant_lin_strike:.1f}°E | امتداد رخنمون (PCA): N{outcrop_strike_az:.1f}°E | SVD RMSE: {rmse_plane:.2f} m")
display(gdf_lines.drop(columns='geometry'))

## 5. Publication-Grade Cartography (300 DPI Maps)
Generates three high-resolution maps saved to `CONFIG['OUT_DIR']`:
1. `Map01_Integrated_4Panel_Dashboard_300DPI.png`
2. `Map02_Structural_Lineaments_and_Rose_300DPI.png`
3. `Map03_Gypsum_Strike_and_100m_Profiles_300DPI.png`

When `CONFIG['ANONYMIZE_MAPS'] = True`, map axes display relative local distances (`m`) instead of absolute UTM coordinates and hide the UTM zone identifier.

In [ ]:
shutil.rmtree(CONFIG['OUT_DIR'], ignore_errors=True)
os.makedirs(CONFIG['OUT_DIR'], exist_ok=True)

plt.rcParams.update({'font.family': 'DejaVu Sans', 'axes.edgecolor': '#2b2d42', 'axes.linewidth': 1.0})
elev_min = np.floor(np.nanmin(dem_arr[inside_mask]) / 20) * 20
elev_max = np.ceil(np.nanmax(dem_arr[inside_mask]) / 20) * 20
contour_lvls = np.arange(elev_min, elev_max + 20, 20)
slope_in = np.where(inside_mask, slope_arr, np.nan)
tpi_in   = np.where(inside_mask, tpi_arr, np.nan)
prio_colors = {'High': '#e63946', 'Medium': '#f4a261', 'Low': '#457b9d'}
sx, sy = outcrop_strike_line.xy

def add_carto_elements(ax, title, pad=105):
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad + 15)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=10.5, fontweight='bold', color='#1d3557', loc='left', pad=8)
    ax.grid(True, linestyle='--', lw=0.4, color='gray', alpha=0.4)

    if CONFIG.get('ANONYMIZE_MAPS', True):
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f'{int(x - minx):,} m'))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, pos: f'{int(y - miny):,} m'))
    else:
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f'{int(x):,} mE'))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, pos: f'{int(y):,} mN'))

    ax.tick_params(axis='x', labelsize=7.5)
    ax.tick_params(axis='y', rotation=90, labelsize=7.5)
    ax.annotate('N', xy=(0.96, 0.93), xytext=(0.96, 0.82), xycoords='axes fraction',
                arrowprops=dict(facecolor='#1d3557', edgecolor='white', width=2.8, headwidth=7.5),
                ha='center', va='center', fontsize=8.5, fontweight='bold', color='#1d3557',
                path_effects=[pe.withStroke(linewidth=2, foreground='white')], zorder=20)
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    bx, by, bh = x0 + (x1 - x0)*0.035, y0 + (y1 - y0)*0.045, (y1 - y0)*0.013
    ax.add_patch(mpatches.Rectangle((bx - 16, by - 10), 232, bh + 38, fc='white', ec='#2b2d42', alpha=0.9, lw=0.7, zorder=19))
    ax.add_patch(mpatches.Rectangle((bx, by), 100, bh, fc='#1d3557', ec='#1d3557', zorder=20))
    ax.add_patch(mpatches.Rectangle((bx + 100, by), 100, bh, fc='white', ec='#1d3557', zorder=20))
    ax.text(bx, by + bh*1.7, '0', fontsize=7, fontweight='bold', ha='center', zorder=21)
    ax.text(bx + 200, by + bh*1.7, '200 m', fontsize=7, fontweight='bold', ha='center', zorder=21)

def label_lines(ax):
    for _, r in gdf_lines.iterrows():
        top_pt = max(r.geometry.coords, key=lambda c: c[1])
        ax.text(top_pt[0], top_pt[1] + 14, r['Line_ID'], fontsize=6.2, fontweight='bold', color='#1d3557',
                ha='center', va='bottom', path_effects=[pe.withStroke(linewidth=2, foreground='white')], zorder=12)

crs_label = "Local Metric Grid (Anonymized)" if CONFIG.get('ANONYMIZE_MAPS', True) else f"{utm_crs.name} ({utm_crs.to_string()})"

# --- نقشه ۱: داشبورد جامع ۴-پانلی ---
fig, axes = plt.subplots(2, 2, figsize=(19, 13.5), dpi=300)
ax1, ax2, ax3, ax4 = axes.flatten()
fig.patch.set_facecolor('white')

ax1.imshow(multi_hillshade, extent=extent, cmap='gray', alpha=0.88, vmin=0.1, vmax=0.95)
slp_b = [0, 15, 25, 35, 65]
slp_cm = ListedColormap(['#2a9d8f', '#e9c46a', '#f4a261', '#d62828'])
im1 = ax1.imshow(slope_in, extent=extent, cmap=slp_cm, norm=BoundaryNorm(slp_b, 4), alpha=0.72)
cb1 = plt.colorbar(im1, ax=ax1, boundaries=slp_b, ticks=[7.5, 20, 30, 50], fraction=0.032, pad=0.02)
cb1.ax.set_yticklabels(['0–15°\nEasy', '15–25°\nMod.', '25–35°\nSteep', '>35°\nHSE!'], fontsize=7)
gdf_utm.boundary.plot(ax=ax1, color='#111111', lw=2.0, zorder=4)
if len(gdf_connectors):
    gdf_connectors.plot(ax=ax1, color='#111111', lw=1.3, ls=':', alpha=0.9, zorder=5)
gdf_lines[gdf_lines['HSE_Flag']=='Safe'].plot(ax=ax1, color='#023e8a', lw=1.8, zorder=6)
gdf_lines[gdf_lines['HSE_Flag']!='Safe'].plot(ax=ax1, color='#111111', lw=1.6, ls='--', zorder=6)
label_lines(ax1)
ax1.legend(handles=[Line2D([0],[0],color='#023e8a',lw=2,label='Safe (≤35°)'),
                    Line2D([0],[0],color='#111111',lw=1.8,ls='--',label='HSE Caution (>35°)'),
                    Line2D([0],[0],color='#111111',lw=1.3,ls=':',label='Profile Connector (Serpentine)')], loc='upper left', fontsize=7.2, framealpha=0.92)
add_carto_elements(ax1, f"A) HSE Terrain Steepness & Serpentine Grid (Az: {traverse_az_deg:.1f}° / {(traverse_az_deg+180)%360:.1f}°)")

ax2.imshow(multi_hillshade, extent=extent, cmap='gray', alpha=0.85, vmin=0.1, vmax=0.95)
im2 = ax2.imshow(tpi_in, extent=extent, cmap='BrBG_r', vmin=-12, vmax=12, alpha=0.78)
plt.colorbar(im2, ax=ax2, fraction=0.032, pad=0.02).set_label('TPI (m) | Teal: Valley | Brown: Ridge', fontsize=7.5)
ax2.contour(xs_g, ys_g, dem_arr, levels=contour_lvls, colors='#264653', linewidths=0.45, alpha=0.5)
gdf_utm.boundary.plot(ax=ax2, color='#111111', lw=2.0, zorder=4)
if len(gdf_connectors):
    gdf_connectors.plot(ax=ax2, color='#1d3557', lw=1.1, ls=':', alpha=0.8, zorder=4)
gdf_lines.plot(ax=ax2, color='#1d3557', lw=1.1, alpha=0.75, zorder=5)
gdf_stations[gdf_stations['TPI_Class']=='Valley/Outcrop'].plot(ax=ax2, color='#ffbe0b', edgecolor='#111111', markersize=26, lw=0.6, zorder=6)
label_lines(ax2)
ax2.legend(handles=[Line2D([0],[0],marker='o',color='w',markerfacecolor='#ffbe0b',markeredgecolor='#111111',markersize=6,label='Valley/Outcrop Station')], loc='upper left', fontsize=7.5, framealpha=0.92)
add_carto_elements(ax2, 'B) Geomorphological Exposure (TPI) & Fresh Outcrop Stations')

ax3.imshow(multi_hillshade, extent=extent, cmap='gray', alpha=0.9, vmin=0.1, vmax=0.95)
im3 = ax3.imshow(combined_gyp, extent=extent, cmap='inferno', vmin=0.15, vmax=0.95, alpha=0.76)
cb3 = plt.colorbar(im3, ax=ax3, fraction=0.032, pad=0.02)
cb3.set_label('Gypsum Prospectivity (0.6 S2 + 0.4 ASTER)', fontsize=7.5)
cb3.ax.axhline(gyp_p80, color='#00f5d4', lw=1.8)
ax3.contour(xs_g, ys_g, gyp_binary, levels=[0.5], colors='#00f5d4', linewidths=1.0, zorder=4)
gdf_utm.boundary.plot(ax=ax3, color='#111111', lw=2.0, zorder=4)
if len(gdf_connectors):
    gdf_connectors.plot(ax=ax3, color='white', lw=2.2, ls=':', alpha=0.85, zorder=5)
    gdf_connectors.plot(ax=ax3, color='#111111', lw=1.2, ls=':', alpha=0.95, zorder=6)
for pr, col in prio_colors.items():
    sub = gdf_lines[gdf_lines['Priority']==pr]
    if len(sub):
        sub.plot(ax=ax3, color='white', lw=3.0, alpha=0.8, zorder=6)
        sub.plot(ax=ax3, color=col, lw=1.7, zorder=7)
ax3.plot(sx, sy, color='#111111', lw=4.2, zorder=8)
ax3.plot(sx, sy, color='#00f5d4', lw=2.2, zorder=9)
gdf_stations[gdf_stations['Target']=='Gypsum Anomaly'].plot(ax=ax3, color='#ffbe0b', edgecolor='#111111', marker='*', markersize=65, lw=0.6, zorder=10)
label_lines(ax3)
ax3.legend(handles=[Line2D([0],[0],color='#00f5d4',lw=2.5,label=f'Outcrop Strike (N{outcrop_strike_az:.1f}°E)'),
                    Line2D([0],[0],color='#e63946',lw=2,label='High Priority'),
                    Line2D([0],[0],color='#f4a261',lw=1.8,label='Medium Priority'),
                    Line2D([0],[0],color='#457b9d',lw=1.6,label='Low Priority'),
                    Line2D([0],[0],color='#111111',lw=1.3,ls=':',label='Serpentine Connector')], loc='upper left', fontsize=7.0, framealpha=0.92)
add_carto_elements(ax3, 'C) Multi-Sensor Gypsum Prospectivity, Outcrop Strike & Continuous Route')

bars = ax4.bar(gdf_lines['Line_ID'], gdf_lines['Total_hr'], color=[prio_colors[p] for p in gdf_lines['Priority']], edgecolor='#1d3557', lw=0.7, alpha=0.9)
ax4.axhline(gdf_lines['Total_hr'].mean(), color='#1d3557', ls=':', lw=1.2)
ax4.set_title('D) Field Logistics: Estimated Survey Time per Line (Tobler + 10 min/Station)', fontsize=10.5, fontweight='bold', color='#1d3557', loc='left', pad=8)
ax4.set_ylabel('Total Field Time (Hours)', fontweight='bold', color='#1d3557')
ax4.set_ylim(0, gdf_lines['Total_hr'].max() * 1.25)
ax4.grid(axis='y', linestyle='--', alpha=0.45)
ax4.tick_params(axis='x', rotation=45, labelsize=7.5)
for bar, (_, r) in zip(bars, gdf_lines.iterrows()):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{r['Total_hr']:.1f}h{'*' if r['HSE_Flag']!='Safe' else ''}", ha='center', va='bottom', fontsize=6.5, fontweight='bold')
ax4.legend(handles=[Line2D([0],[0],color='#e63946',lw=6,label='High'), Line2D([0],[0],color='#f4a261',lw=6,label='Medium'),
                    Line2D([0],[0],color='#457b9d',lw=6,label='Low (* = HSE >35°)')], loc='upper right', fontsize=7.8, framealpha=0.92)

fig.suptitle('GYPSUM EXPLORATION — INTEGRATED MULTI-SENSOR PROSPECTIVITY, HSE & SERPENTINE 3D TRAVERSE PLAN', fontsize=13.5, fontweight='bold', color='#1d3557', y=0.985)
fig.text(0.5, 0.01, f"CRS: {crs_label} | Area: {area_km2:.4f} km² | Spacing: {CONFIG['LINE_SPACING_M']}m | "
                    f"Lines: {len(gdf_lines)}, Connectors: {len(gdf_connectors)} ({conn_len_km:.2f} km), Stations: {len(gdf_stations)} | "
                    f"Data: Copernicus DEM 2024 (30m), Sentinel-2 SR (10/20m) & ASTER TIR (90m bilinear)",
         ha='center', fontsize=8, style='italic', bbox=dict(boxstyle='round,pad=0.35', fc='#f8f9fa', ec='#adb5bd'))
plt.tight_layout(rect=[0, 0.035, 1, 0.96])
plt.savefig(f"{CONFIG['OUT_DIR']}/Map01_Integrated_4Panel_Dashboard_300DPI.png", dpi=300)
plt.show()

# --- نقشه ۲: تحلیل ساختاری و گل‌سرخی دوجهته ---
fig = plt.figure(figsize=(16, 7), dpi=300)
fig.patch.set_facecolor('white')
ax_m = fig.add_subplot(1, 2, 1)
ax_r = fig.add_subplot(1, 2, 2, projection='polar')
ax_m.imshow(multi_hillshade, extent=extent, cmap='gray', alpha=0.9, vmin=0.1, vmax=0.95)
cs = ax_m.contour(xs_g, ys_g, dem_arr, levels=contour_lvls, colors='#0077b6', linewidths=0.55, alpha=0.6)
plt.setp(ax_m.clabel(cs, inline=True, fontsize=6.5, fmt='%d m'), path_effects=[pe.withStroke(linewidth=1.5, foreground='white')])
ax_m.contour(xs_g, ys_g, gyp_binary, levels=[0.5], colors='#ff006e', linewidths=1.3)
gdf_utm.boundary.plot(ax=ax_m, color='#111111', lw=2.2, zorder=5)
if lineament_geoms:
    gpd.GeoSeries(lineament_geoms, crs=utm_crs).plot(ax=ax_m, color='#111111', lw=2.5, zorder=6)
    gpd.GeoSeries(lineament_geoms, crs=utm_crs).plot(ax=ax_m, color='#ffbe0b', lw=1.3, zorder=7)
ax_m.legend(handles=[Line2D([0],[0],color='#ffbe0b',lw=2,label=f'Lineaments (n={len(lineament_geoms)})'),
                     Line2D([0],[0],color='#ff006e',lw=1.5,label='Gypsum Contact (≥P80)')], loc='upper left', fontsize=7.5, framealpha=0.92)
add_carto_elements(ax_m, 'A) Structural Lineaments, 20m Contours & Gypsum Contact')

if lineament_azimuths:
    bins = np.arange(0, 181, 15)
    hist, edges_b = np.histogram(lineament_azimuths, bins=bins, weights=lineament_lengths)
    angles = np.deg2rad(np.concatenate([edges_b[:-1], edges_b[:-1] + 180]))
    vals = np.concatenate([hist, hist])
    ax_r.set_theta_zero_location('N'); ax_r.set_theta_direction(-1)
    ax_r.bar(angles, vals, width=np.deg2rad(13.5), align='edge', color='#e63946', edgecolor='#1d3557', lw=0.8, alpha=0.82)
    rmax = np.max(vals) * 1.15; ax_r.set_ylim(0, rmax)
    ax_r.plot([np.deg2rad(dominant_lin_strike), np.deg2rad(dominant_lin_strike)+np.pi], [rmax, rmax], color='#1d3557', lw=2.2, label=f'Lineament Strike (N{dominant_lin_strike:.1f}°E)')
    ax_r.plot([np.deg2rad(traverse_az_deg), np.deg2rad(traverse_az_deg)+np.pi], [rmax, rmax], color='#2a9d8f', lw=2.0, ls='--', label=f'Traverse Az ({traverse_az_deg:.1f}°)')
    ax_r.legend(loc='lower right', bbox_to_anchor=(1.22, -0.08), fontsize=8, framealpha=0.92)
    ax_r.set_title(f"B) Structural Rose Diagram (Strike: N{dominant_lin_strike:.1f}°E | SVD RMSE: {rmse_plane:.1f}m)", fontsize=10.5, fontweight='bold', color='#1d3557', pad=18)
plt.tight_layout()
plt.savefig(f"{CONFIG['OUT_DIR']}/Map02_Structural_Lineaments_and_Rose_300DPI.png", dpi=300)
plt.show()

# --- نقشه ۳: نقشه اختصاصی امتداد رخنمون گچ و پروفیل‌های پیوسته زیگزاگ ---
fig, ax = plt.subplots(figsize=(14, 9.2), dpi=300)
fig.patch.set_facecolor('white')
ax.imshow(multi_hillshade, extent=extent, cmap='gray', alpha=0.9, vmin=0.1, vmax=0.95)
im_p = ax.imshow(combined_gyp, extent=extent, cmap='inferno', alpha=0.72, vmin=0.15, vmax=0.95)
cs3 = ax.contour(xs_g, ys_g, dem_arr, levels=contour_lvls, colors='#264653', linewidths=0.5, alpha=0.55)
plt.setp(ax.clabel(cs3, inline=True, fontsize=6.5, fmt='%d m'), path_effects=[pe.withStroke(linewidth=1.5, foreground='white')])
ax.contour(xs_g, ys_g, gyp_binary, levels=[0.5], colors='#ffbe0b', linewidths=1.1)
gdf_utm.boundary.plot(ax=ax, color='#111111', lw=2.2, zorder=5)
if len(gdf_connectors):
    gdf_connectors.plot(ax=ax, color='white', lw=2.4, ls=':', alpha=0.9, zorder=6)
    gdf_connectors.plot(ax=ax, color='#111111', lw=1.4, ls=':', alpha=0.95, zorder=7)
for pr, col in prio_colors.items():
    sub = gdf_lines[gdf_lines['Priority']==pr]
    if len(sub):
        sub.plot(ax=ax, color='white', lw=3.2, alpha=0.85, zorder=7)
        sub.plot(ax=ax, color=col, lw=1.8, zorder=8)
ax.plot(sx, sy, color='#111111', lw=5.0, zorder=9)
ax.plot(sx, sy, color='#00f5d4', lw=2.6, zorder=10)
label_lines(ax)
cbar = fig.colorbar(im_p, ax=ax, fraction=0.028, pad=0.015, shrink=0.8)
cbar.set_label('Integrated Gypsum Prospectivity Index (0.6 S2 + 0.4 ASTER)', fontsize=8.5, fontweight='bold')
cbar.ax.axhline(gyp_p80, color='#ffbe0b', lw=1.8)
ax.legend(handles=[
    Line2D([0],[0],color='#00f5d4',lw=3.0,label=f'Gypsum Outcrop Strike Axis (N{outcrop_strike_az:.1f}°E)'),
    Line2D([0],[0],color='#e63946',lw=2.0,label='100m Profile — High Priority (≥20%)'),
    Line2D([0],[0],color='#f4a261',lw=1.8,label='100m Profile — Medium Priority (10–20%)'),
    Line2D([0],[0],color='#457b9d',lw=1.6,label='100m Profile — Low Priority (<10%)'),
    Line2D([0],[0],color='#111111',lw=1.5,ls=':',label='Serpentine Profile Connector'),
    Line2D([0],[0],color='#ffbe0b',lw=1.3,label=f'Gypsum Anomaly Boundary (≥P80={gyp_p80:.2f})')
], loc='upper left', fontsize=7.6, framealpha=0.93)
add_carto_elements(ax, f"Gypsum Outcrop Strike Axis & Continuous 100m Serpentine Profiles ({len(gdf_lines)} Lines)")
plt.tight_layout()
plt.savefig(f"{CONFIG['OUT_DIR']}/Map03_Gypsum_Strike_and_100m_Profiles_300DPI.png", dpi=300)
plt.show()